# Gerenciador de Assets Visuais — SEGAPE/MEC

Ferramenta interativa para adicionar imagens ao repositório e gerar fórmulas para o Looker Studio.

## 1. Configuração inicial

Execute a célula abaixo para instalar as dependências necessárias.

In [ ]:
# Instalação das dependências
!pip install unidecode requests --quiet

## 2. Selecionar e enviar arquivo

Clique no botão abaixo para selecionar a imagem que deseja adicionar ao repositório.

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import base64
import json
from pathlib import Path

try:
    from unidecode import unidecode
except ImportError:
    # Fallback simples se unidecode não estiver disponível
    def unidecode(texto):
        return texto

# Tentativa de import para Google Colab / BigQuery Notebook
try:
    from google.colab import files as colab_files
    AMBIENTE = "colab"
except ImportError:
    try:
        # Ambiente Jupyter local com ipywidgets
        import ipywidgets as widgets
        from IPython.display import display, HTML, clear_output
        AMBIENTE = "jupyter"
    except ImportError:
        AMBIENTE = "terminal"

# --- Constantes ---
REPOSITORIO = "SEGAPE/imagens_ux_ui"
BRANCH = "main"
URL_CDN_BASE = "https://cdn.jsdelivr.net/gh/{repo}@{branch}"
GITHUB_API_BASE = "https://api.github.com"
TIPOS_VALIDOS = ["logo", "icon"]

# --- Funções auxiliares ---
def sanitizar_nome(texto):
    """Remove acentos, converte para snake_case limpo."""
    resultado = unidecode(texto)
    resultado = resultado.lower()
    resultado = re.sub(r"[\s\-\.]+", "_", resultado)
    resultado = re.sub(r"[^a-z0-9_]", "_", resultado)
    resultado = re.sub(r"_+", "_", resultado)
    return resultado.strip("_")

def montar_nome_arquivo(programa, tipo, variante, extensao):
    """Monta nome padronizado: {programa}_{tipo}[_{variante}].{extensao}"""
    programa_san = sanitizar_nome(programa)
    tipo_san = sanitizar_nome(tipo)
    extensao = extensao.lstrip(".")
    if variante:
        variante_san = sanitizar_nome(variante)
        return f"{programa_san}_{tipo_san}_{variante_san}.{extensao}"
    return f"{programa_san}_{tipo_san}.{extensao}"

def gerar_url_cdn(caminho_relativo):
    """Retorna URL jsDelivr CDN para o caminho."""
    base = URL_CDN_BASE.format(repo=REPOSITORIO, branch=BRANCH)
    return f"{base}/{caminho_relativo}"

# --- Estado global do notebook ---
arquivos_enviados = []

def upload_arquivo():
    """Gerencia o upload do arquivo conforme o ambiente."""
    global arquivos_enviados

    if AMBIENTE == "colab":
        resultado = colab_files.upload()
        for nome, conteudo in resultado.items():
            extensao = Path(nome).suffix.lower()
            if not extensao:
                extensao = ".png"  # fallback
            arquivos_enviados.append({
                "nome_original": nome,
                "conteudo": conteudo,
                "extensao": extensao,
            })
            print(f"Arquivo recebido: {nome}")

    elif AMBIENTE == "jupyter":
        uploader = widgets.FileUpload(
            accept="image/*",
            multiple=True,
            description="Selecionar imagem",
        )
        botao = widgets.Button(
            description="Confirmar seleção",
            button_style="success",
        )
        saida = widgets.Output()

        def ao_confirmar(_):
            with saida:
                clear_output()
                if not uploader.value:
                    print("Nenhum arquivo selecionado.")
                    return
                for item in uploader.value:
                    nome = item.name if hasattr(item, 'name') else item['name']
                    conteudo = item.content if hasattr(item, 'content') else item['content']
                    extensao = Path(nome).suffix.lower()
                    if not extensao:
                        extensao = ".png"
                    arquivos_enviados.append({
                        "nome_original": nome,
                        "conteudo": bytes(conteudo),
                        "extensao": extensao,
                    })
                    print(f"Arquivo recebido: {nome}")

        botao.on_click(ao_confirmar)
        display(widgets.VBox([uploader, botao, saida]))

    else:
        caminho = input("Caminho do arquivo de imagem: ").strip()
        if not caminho or not Path(caminho).exists():
            print("Arquivo não encontrado.")
            return
        with open(caminho, "rb") as f:
            conteudo = f.read()
        nome = Path(caminho).name
        extensao = Path(caminho).suffix.lower()
        if not extensao:
            extensao = ".png"
        arquivos_enviados.append({
            "nome_original": nome,
            "conteudo": conteudo,
            "extensao": extensao,
        })
        print(f"Arquivo recebido: {nome}")

print(f"Ambiente detectado: {AMBIENTE}")
upload_arquivo()

## 3. Classificar os arquivos

Preencha o programa, tipo e variante para cada arquivo enviado.

In [ ]:
# Classificação interativa dos arquivos enviados
assets_classificados = []

for i, arq in enumerate(arquivos_enviados):
    print(f"\n--- Arquivo {i+1}/{len(arquivos_enviados)}: {arq['nome_original']} ---")
    programa = input("Programa (ex: cnca, painel_ministro): ").strip()
    tipo = ""
    while tipo not in TIPOS_VALIDOS:
        tipo = input("Tipo (logo/icon): ").strip().lower()
        if tipo not in TIPOS_VALIDOS:
            print(f"  Tipo inválido. Escolha entre: {', '.join(TIPOS_VALIDOS)}")
    variante = input("Variante (opcional, ex: bronze, escuro \u2014 deixe vazio se n\u00e3o houver): ").strip() or None

    nome_final = montar_nome_arquivo(programa, tipo, variante, arq["extensao"])
    programa_san = sanitizar_nome(programa)
    caminho_repo = f"assets/{programa_san}/{nome_final}"
    url_cdn = gerar_url_cdn(caminho_repo)

    assets_classificados.append({
        "nome_original": arq["nome_original"],
        "conteudo": arq["conteudo"],
        "programa": programa_san,
        "tipo": tipo,
        "variante": sanitizar_nome(variante) if variante else None,
        "nome_final": nome_final,
        "caminho_repo": caminho_repo,
        "url_cdn": url_cdn,
    })

    print(f"  Nome padronizado: {caminho_repo}")
    print(f"  URL CDN: {url_cdn}")

print(f"\n{len(assets_classificados)} arquivo(s) classificado(s).")

## 4. Enviar para o GitHub

Informe seu token de acesso pessoal (PAT) do GitHub com permissão de escrita no repositório.
Se preferir, use o OAuth Device Flow executando o script `gerenciador_bigquery.py`.

In [ ]:
# Upload para o GitHub via API
import requests

token = input("Token GitHub (PAT com permissão repo): ").strip()

if not token:
    print("Token não informado. Upload cancelado.")
else:
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
    }

    enviados = 0
    for asset in assets_classificados:
        caminho = asset["caminho_repo"]
        url_api = f"{GITHUB_API_BASE}/repos/{REPOSITORIO}/contents/{caminho}"

        # Verificar se já existe (para obter SHA)
        resp_get = requests.get(url_api, headers=headers)
        sha = None
        if resp_get.status_code == 200:
            sha = resp_get.json().get("sha")

        conteudo_b64 = base64.b64encode(asset["conteudo"]).decode("utf-8")
        dados = {
            "message": f"feat: adiciona {asset['nome_final']}",
            "content": conteudo_b64,
            "branch": BRANCH,
        }
        if sha:
            dados["sha"] = sha

        resp_put = requests.put(url_api, headers=headers, json=dados)
        if resp_put.status_code in (200, 201):
            print(f"Enviado: {caminho}")
            enviados += 1
        else:
            print(f"Erro ao enviar {caminho}: {resp_put.status_code} \u2014 {resp_put.json().get('message', '')}")

    print(f"\n{enviados}/{len(assets_classificados)} arquivo(s) enviado(s) com sucesso.")

## 5. Gerar fórmula para o Looker Studio

Preencha as informações abaixo para gerar a fórmula CASE/WHEN pronta para colar no Looker Studio.
Você pode usar tanto os arquivos que acabou de enviar quanto qualquer asset já existente no catálogo.

In [ ]:
# Gerador de fórmula CASE/WHEN para o Looker Studio
print("=== Gerador de fórmula CASE/WHEN ===")
print()

nome_campo = input("Nome do campo condicional (ex: selo, tipo_escola): ").strip()
if not nome_campo:
    print("Nome do campo obrigatório.")
else:
    linhas = []
    print(f"\nAgora defina as linhas da fórmula.")
    print(f"Para cada linha, informe o valor do campo '{nome_campo}' e a URL da imagem.")
    print(f"Digite 'fim' no valor para encerrar.\n")

    # Mostrar assets disponíveis
    print("--- Assets disponíveis (enviados nesta sessão) ---")
    for i, asset in enumerate(assets_classificados):
        ident = f"{asset['programa']}/{asset['tipo']}"
        if asset['variante']:
            ident += f"/{asset['variante']}"
        print(f"  [{i+1}] {ident} \u2014 {asset['url_cdn']}")
    print()

    # Tentar carregar catálogo existente se disponível
    catalogo_path = Path("catalogo.json")
    assets_catalogo = []
    if catalogo_path.exists():
        with open(catalogo_path, "r", encoding="utf-8") as f:
            assets_catalogo = json.load(f)
        print("--- Assets disponíveis (catálogo existente) ---")
        for i, asset in enumerate(assets_catalogo):
            ident = f"{asset.get('programa', '?')}/{asset.get('tipo', '?')}"
            var = asset.get("variante")
            if var:
                ident += f"/{var}"
            print(f"  [c{i+1}] {ident} \u2014 {asset.get('url_cdn', '')}")
        print()

    while True:
        valor = input(f"Valor de '{nome_campo}' (ou 'fim'): ").strip()
        if valor.lower() == "fim":
            break

        url = input(f"  URL da imagem (ou n\u00famero do asset [1], [c1], etc.): ").strip()

        # Resolver referência a asset
        if url.startswith("c") and url[1:].isdigit():
            idx = int(url[1:]) - 1
            if 0 <= idx < len(assets_catalogo):
                url = assets_catalogo[idx].get("url_cdn", "")
            else:
                print("  \u00cdndice inv\u00e1lido.")
                continue
        elif url.isdigit():
            idx = int(url) - 1
            if 0 <= idx < len(assets_classificados):
                url = assets_classificados[idx]["url_cdn"]
            else:
                print("  \u00cdndice inv\u00e1lido.")
                continue

        linhas.append((valor, url))
        print(f"  Adicionado: WHEN {nome_campo} = \"{valor}\" THEN \"{url}\"")

    if linhas:
        print("\n=== F\u00f3rmula gerada ===\n")
        formula = "CASE\n"
        for valor, url in linhas:
            formula += f'  WHEN {nome_campo} = "{valor}" THEN "{url}"\n'
        formula += f"  WHEN {nome_campo} IS NULL THEN NULL\n"
        formula += "  ELSE NULL\n"
        formula += "END"
        print(formula)
        print("\nCopie a f\u00f3rmula acima e cole no campo calculado do Looker Studio.")
    else:
        print("Nenhuma linha adicionada.")

## Pronto!

A fórmula gerada pode ser colada diretamente no campo calculado do Looker Studio.
Lembre-se de configurar o tipo do campo como **Imagem** e usar o componente **Tabela** para exibir.